# Corrective Networks

A **corrective network** (compensator) is a dynamical block inserted in the forward path of a closed-loop system to reshape the open-loop frequency response, adjusting phase margin, bandwidth, or steady-state accuracy.

Two complementary compensators share the same pair of break frequencies $\omega_1 = 1/\tau$ and $\omega_2 = 1/(\alpha\tau)$, differing only in which factor sits in the numerator:

| | **Lead** $R_{lead}(s)$ | **Lag** $R_{lag}(s)$ |
|---|---|---|
| Transfer function | $\dfrac{1+s\tau}{1+s\alpha\tau}$ | $\dfrac{1+s\alpha\tau}{1+s\tau}$ |
| Constraint | $0 < \alpha < 1$ | $0 < \alpha < 1$ |
| Phase effect | advance $\phi > 0$ | lag $\phi < 0$ |
| HF magnitude | $+20\log_{10}(1/\alpha)$ dB | $20\log_{10}\alpha$ dB (attenuation) |
| Extremum at | $\omega_m = 1/(\tau\sqrt{\alpha})$ | $\omega_m = 1/(\tau\sqrt{\alpha})$ |
| Primary goal | $\uparrow$ phase margin, faster response | $\uparrow$ low-freq loop gain, noise rejection |

The plant used throughout is $G(s) = \dfrac{K}{s\,(s+a)}$ — a type-1 second-order system with gain $K$ and pole $a$, both adjustable via sliders.

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import TransferFunction, bode, step
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import clear_output

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
})


def _polyadd(a, b):
    """Add two polynomial coefficient arrays (high-degree first) of any lengths."""
    if len(a) < len(b):
        a = np.concatenate([np.zeros(len(b) - len(a)), a])
    elif len(b) < len(a):
        b = np.concatenate([np.zeros(len(a) - len(b)), b])
    return a + b


def _closed_loop(num_L, den_L):
    """Closed-loop coefficients for unity negative feedback: T = L/(1+L)."""
    return num_L, _polyadd(den_L, num_L)


def _phase_margin(num, den, w):
    """Gain crossover frequency and phase margin. Returns (NaN, NaN) if no 0-dB crossing."""
    _, mag_dB, ph_deg = bode(TransferFunction(num, den), w=w)
    cross = np.where(np.diff(np.sign(mag_dB)))[0]
    if len(cross) == 0:
        return np.nan, np.nan
    i = cross[0]
    f = mag_dB[i] / (mag_dB[i] - mag_dB[i + 1])
    wc = np.exp(np.log(w[i]) + f * (np.log(w[i + 1]) - np.log(w[i])))
    ph_c = ph_deg[i] + f * (ph_deg[i + 1] - ph_deg[i])
    return wc, 180.0 + ph_c


def _step_safe(num, den, t):
    """Step response; returns NaN array for unstable systems."""
    if np.any(np.real(np.roots(den)) > 1e-6):
        return np.full_like(t, np.nan)
    try:
        _, y = step(TransferFunction(num, den), T=t)
        return y
    except Exception:
        return np.full_like(t, np.nan)


def _draw(num_R, den_R, num_L, den_L, num_CL0, den_CL0, num_CL, den_CL,
          w, t, title, color, w1, w2, wm, phi_ext, hf_label):
    """2x2 figure: compensator Bode (left) | open-loop Bode + step response (right)."""
    _, mag_R, ph_R = bode(TransferFunction(num_R, den_R), w=w)
    _, mag_L, _    = bode(TransferFunction(num_L, den_L), w=w)
    wc, PM = _phase_margin(num_L, den_L, w)
    y0 = _step_safe(num_CL0, den_CL0, t)
    yc = _step_safe(num_CL,  den_CL,  t)

    pm_str = f'PM = {PM:.1f}°' if not np.isnan(PM) else 'no crossover'
    clear_output(wait=True)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle(f'{title}   |   {pm_str}', fontsize=11, fontweight='bold')

    # Compensator magnitude
    ax = axes[0, 0]
    ax.semilogx(w, mag_R, color=color, lw=2)
    ax.axvline(w1, color='green',      ls='--', lw=1.2,
               label=fr'$\omega_1 = 1/\tau = {w1:.2f}$')
    ax.axvline(w2, color='darkorange', ls='--', lw=1.2,
               label=fr'$\omega_2 = 1/\alpha\tau = {w2:.2f}$')
    ax.axvline(wm, color='purple',     ls=':',  lw=1.8,
               label=fr'$\omega_m = {wm:.2f}$')
    ax.axhline(0, color='k', ls='--', lw=0.7)
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'$R(s)$ magnitude  —  HF: {hf_label}')
    ax.legend(); ax.grid(True, which='both', alpha=0.3)

    # Compensator phase
    ax = axes[1, 0]
    ax.semilogx(w, ph_R, color=color, lw=2)
    ax.axvline(wm, color='purple', ls=':', lw=1.8,
               label=fr'$\omega_m$,   $\phi_{{ext}} = {phi_ext:.1f}°$')
    ax.axhline(0, color='k', ls='--', lw=0.7)
    ax.set_xlabel('Frequency (rad/s)'); ax.set_ylabel('Phase (deg)')
    ax.set_title('$R(s)$ phase')
    ax.legend(); ax.grid(True, which='both', alpha=0.3)

    # Open-loop magnitude with phase margin
    ax = axes[0, 1]
    ax.semilogx(w, mag_L, color=color, lw=2, label=r'$L(s) = R(s)\cdot G(s)$')
    ax.axhline(0, color='k', ls='--', lw=0.8)
    if not np.isnan(PM):
        ax.axvline(wc, color='gray', ls='--', lw=1.2,
                   label=fr'$\omega_c = {wc:.2f}$,   {pm_str}')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title('Open-loop $L(s)$')
    ax.legend(); ax.grid(True, which='both', alpha=0.3)

    # Closed-loop step response comparison
    ax = axes[1, 1]
    ax.plot(t, y0, 'k--', lw=1.5, label='Uncompensated $G(s)$')
    ax.plot(t, yc, color=color, lw=2.2, label='With $R(s)$')
    ax.axhline(1, color='gray', ls=':', lw=0.8)
    valid = yc[~np.isnan(yc)]
    ylim_top = max(2.0, float(np.max(valid)) * 1.25) if len(valid) else 2.0
    ax.set_ylim([0, ylim_top])
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Output')
    ax.set_title('Closed-loop step response')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## Lead Compensator

$$R_{lead}(s) = \frac{1+s\tau}{1+s\alpha\tau}, \qquad 0 < \alpha < 1$$

**Bode asymptotes** — the zero at $\omega_1 = 1/\tau$ opens a $+20\,\text{dB/dec}$ ramp; the pole at $\omega_2 = 1/(\alpha\tau) > \omega_1$ closes it back to flat. The asymptotic plateau is $-20\log_{10}\alpha > 0\,\text{dB}$. Phase is positive throughout $[\omega_1,\,\omega_2]$, reaching its peak at the geometric mean of the two break frequencies:

$$\omega_m = \frac{1}{\tau\sqrt{\alpha}}, \qquad \phi_{max} = \arcsin\!\left(\frac{1-\alpha}{1+\alpha}\right)$$

**Design rationale** — align $\omega_m$ with the desired gain-crossover frequency $\omega_c$ of $G(s)$. The phase advance $\phi_{max}$ is directly added to the phase margin. The simultaneous gain boost shifts $\omega_c$ to higher frequency, yielding wider bandwidth and a faster closed-loop step response.

**Trade-off** — smaller $\alpha$ gives more phase advance but amplifies high-frequency noise by $1/\alpha$. A practical lower bound is $\alpha \approx 0.05$ ($\phi_{max} \approx 65\degree$, gain $\approx 26\,\text{dB}$).

In [ ]:
def plot_lead(alpha, tau, K, a):
    w = np.logspace(-2, 3, 2000)
    num_R = np.array([tau, 1.0])
    den_R = np.array([alpha * tau, 1.0])
    num_G = np.array([float(K)])
    den_G = np.array([1.0, float(a), 0.0])
    num_L  = np.polymul(num_R, num_G)
    den_L  = np.polymul(den_R, den_G)
    num_CL,  den_CL  = _closed_loop(num_L, den_L)
    num_CL0, den_CL0 = _closed_loop(num_G, den_G)
    t    = np.linspace(0, min(60.0, 40.0 / a), 2000)
    w1   = 1.0 / tau
    w2   = 1.0 / (alpha * tau)
    wm   = 1.0 / (tau * np.sqrt(alpha))
    phi  = np.degrees(np.arcsin((1.0 - alpha) / (1.0 + alpha)))
    hflabel = fr'+{-20*np.log10(alpha):.1f} dB'
    title = (
        fr'Lead  $R_{{lead}}$  '
        fr'|  α={alpha:.2f},  τ={tau:.1f},  K={K:.1f},  a={a:.1f}'
    )
    _draw(num_R, den_R, num_L, den_L, num_CL0, den_CL0, num_CL, den_CL,
          w, t, title, 'steelblue', w1, w2, wm, phi, hflabel)


interact(
    plot_lead,
    alpha=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.3,
                              description='α:',
                              style={'description_width': 'initial'}),
    tau  =widgets.FloatSlider(min=0.1,  max=10,   step=0.1,  value=1.0,
                              description='τ:',
                              style={'description_width': 'initial'}),
    K    =widgets.FloatSlider(min=0.5,  max=20,   step=0.5,  value=5.0,
                              description='K (plant):',
                              style={'description_width': 'initial'}),
    a    =widgets.FloatSlider(min=0.1,  max=5.0,  step=0.1,  value=1.0,
                              description='a (plant):',
                              style={'description_width': 'initial'}),
)


interactive(children=(FloatSlider(value=0.3, description='α:', max=0.95, min=0.05, step=0.05, style=SliderStyl…

<function __main__.plot_lead(alpha, tau, K, a)>

## Lag Compensator

$$R_{lag}(s) = \frac{1+s\alpha\tau}{1+s\tau}, \qquad 0 < \alpha < 1$$

**Bode asymptotes** — the pole at $\omega_1 = 1/\tau$ opens a $-20\,\text{dB/dec}$ slope; the zero at $\omega_2 = 1/(\alpha\tau) > \omega_1$ closes it back to flat. The asymptotic attenuation floor is $20\log_{10}\alpha < 0\,\text{dB}$. Phase is negative throughout $[\omega_1,\,\omega_2]$, reaching its trough at:

$$\omega_m = \frac{1}{\tau\sqrt{\alpha}}, \qquad \phi_{min} = -\arcsin\!\left(\frac{1-\alpha}{1+\alpha}\right)$$

**Design rationale** — place $\omega_2 = 1/(\alpha\tau)$ at least one decade *below* the target crossover $\omega_c$ so that phase erosion at crossover is negligible ($< 5\degree$). The attenuation band then reduces the high-frequency loop gain, allowing the loop gain $K$ to be raised (improving steady-state accuracy) while keeping the phase margin acceptable.

**Trade-off** — if $\tau$ is too small, $\omega_m$ falls near $\omega_c$ and the lag erodes phase margin, potentially destabilising the loop. Increase $\tau$ to push $\omega_m$ well below $\omega_c$.

In [ ]:
def plot_lag(alpha, tau, K, a):
    w = np.logspace(-2, 3, 2000)
    num_R = np.array([alpha * tau, 1.0])
    den_R = np.array([tau, 1.0])
    num_G = np.array([float(K)])
    den_G = np.array([1.0, float(a), 0.0])
    num_L  = np.polymul(num_R, num_G)
    den_L  = np.polymul(den_R, den_G)
    num_CL,  den_CL  = _closed_loop(num_L, den_L)
    num_CL0, den_CL0 = _closed_loop(num_G, den_G)
    t    = np.linspace(0, min(100.0, 80.0 / a), 2000)
    w1   = 1.0 / tau
    w2   = 1.0 / (alpha * tau)
    wm   = 1.0 / (tau * np.sqrt(alpha))
    phi  = -np.degrees(np.arcsin((1.0 - alpha) / (1.0 + alpha)))
    hflabel = fr'{20*np.log10(alpha):.1f} dB'
    title = (
        fr'Lag  $R_{{lag}}$  '
        fr'|  α={alpha:.2f},  τ={tau:.1f},  K={K:.1f},  a={a:.1f}'
    )
    _draw(num_R, den_R, num_L, den_L, num_CL0, den_CL0, num_CL, den_CL,
          w, t, title, 'teal', w1, w2, wm, phi, hflabel)


interact(
    plot_lag,
    alpha=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.1,
                              description='α:',
                              style={'description_width': 'initial'}),
    tau  =widgets.FloatSlider(min=0.5,  max=30,   step=0.5,  value=10.0,
                              description='τ:',
                              style={'description_width': 'initial'}),
    K    =widgets.FloatSlider(min=0.5,  max=20,   step=0.5,  value=5.0,
                              description='K (plant):',
                              style={'description_width': 'initial'}),
    a    =widgets.FloatSlider(min=0.1,  max=5.0,  step=0.1,  value=1.0,
                              description='a (plant):',
                              style={'description_width': 'initial'}),
)


interactive(children=(FloatSlider(value=0.1, description='α:', max=0.95, min=0.05, step=0.05, style=SliderStyl…

<function __main__.plot_lag(alpha, tau, K, a)>